# Netflix Data Analysis — SQL

## Project Overview

This notebook performs SQL-based analysis on the cleaned Netflix
dataset using SQLite and Python. The analysis focuses on content
distribution, ratings, release trends, countries, genres, directors,
and other business-oriented insights.

### Tools Used
- Python
- SQLite
- SQL
- Pandas
- Jupyter Notebook

In [1]:
import pandas as pd
import sqlite3

In [2]:
df = pd.read_csv("netflix_cleaned.csv")

print("Dataset loaded successfully!")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

Dataset loaded successfully!
Rows: 8807
Columns: 14


In [3]:
df.head()

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description,year_added,month_added
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,Unknown,United States,2021-09-25,2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm...",2021.0,September
1,s2,TV Show,Blood & Water,Unknown,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,2021-09-24,2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t...",2021.0,September
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",Unknown,2021-09-24,2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...,2021.0,September
3,s4,TV Show,Jailbirds New Orleans,Unknown,Unknown,Unknown,2021-09-24,2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo...",2021.0,September
4,s5,TV Show,Kota Factory,Unknown,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,2021-09-24,2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...,2021.0,September


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8807 entries, 0 to 8806
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   show_id       8807 non-null   str    
 1   type          8807 non-null   str    
 2   title         8807 non-null   str    
 3   director      8807 non-null   str    
 4   cast          8807 non-null   str    
 5   country       8807 non-null   str    
 6   date_added    8797 non-null   str    
 7   release_year  8807 non-null   int64  
 8   rating        8807 non-null   str    
 9   duration      8807 non-null   str    
 10  listed_in     8807 non-null   str    
 11  description   8807 non-null   str    
 12  year_added    8797 non-null   float64
 13  month_added   8797 non-null   str    
dtypes: float64(1), int64(1), str(12)
memory usage: 963.4 KB


In [5]:
df.columns

Index(['show_id', 'type', 'title', 'director', 'cast', 'country', 'date_added',
       'release_year', 'rating', 'duration', 'listed_in', 'description',
       'year_added', 'month_added'],
      dtype='str')

In [6]:
conn = sqlite3.connect("netflix_analysis.db")

print("SQLite database connected successfully!")

SQLite database connected successfully!


In [7]:
df.to_sql(
    "netflix_titles",
    conn,
    if_exists="replace",
    index=False
)

print("Netflix data loaded into SQL table successfully!")

Netflix data loaded into SQL table successfully!


## 1. Total Number of Netflix Titles

In [8]:
query = """
SELECT COUNT(*) AS total_titles
FROM netflix_titles;
"""

pd.read_sql_query(query, conn)

,total_titles
0,8807


## 2. Movies vs TV Shows

In [9]:
query = """
SELECT
    type,
    COUNT(*) AS total_titles
FROM netflix_titles
GROUP BY type
ORDER BY total_titles DESC;
"""

pd.read_sql_query(query, conn)

,type,total_titles
0,Movie,6131
1,TV Show,2676


## 3. Content by Release Year

In [10]:
query = """
SELECT
    release_year,
    COUNT(*) AS total_titles
FROM netflix_titles
GROUP BY release_year
ORDER BY release_year;
"""

pd.read_sql_query(query, conn)

,release_year,total_titles
0,1925,1
1,1942,2
2,1943,3
3,1944,3
4,1945,4
...,...,...
69,2017,1032
70,2018,1147
71,2019,1030
72,2020,953


## 4. Top 10 Release Years

In [11]:
query = """
SELECT
    release_year,
    COUNT(*) AS total_titles
FROM netflix_titles
GROUP BY release_year
ORDER BY total_titles DESC
LIMIT 10;
"""

pd.read_sql_query(query, conn)

,release_year,total_titles
0,2018,1147
1,2017,1032
2,2019,1030
3,2020,953
4,2016,902
5,2021,592
6,2015,560
7,2014,352
8,2013,288
9,2012,237


## 5. Content Distribution by Rating

In [12]:
query = """
SELECT
    rating,
    COUNT(*) AS total_titles
FROM netflix_titles
GROUP BY rating
ORDER BY total_titles DESC;
"""

pd.read_sql_query(query, conn)

,rating,total_titles
0,TV-MA,3207
1,TV-14,2160
2,TV-PG,863
3,R,799
4,PG-13,490
5,TV-Y7,334
6,TV-Y,307
7,PG,287
8,TV-G,220
9,NR,80


## 6. Top 10 Directors by Number of Titles

In [13]:
query = """
SELECT
    director,
    COUNT(*) AS total_titles
FROM netflix_titles
WHERE director != 'Unknown'
GROUP BY director
ORDER BY total_titles DESC
LIMIT 10;
"""

pd.read_sql_query(query, conn)

,director,total_titles
0,Rajiv Chilaka,19
1,"Raúl Campos, Jan Suter",18
2,Suhas Kadav,16
3,Marcus Raboy,16
4,Jay Karas,14
5,Cathy Garcia-Molina,13
6,Youssef Chahine,12
7,Martin Scorsese,12
8,Jay Chapman,12
9,Steven Spielberg,11


## 7. Content Distribution by Country

In [14]:
countries_df = df[["show_id", "country"]].copy()

countries_df["country"] = countries_df["country"].str.split(", ")

countries_df = countries_df.explode("country")

countries_df = countries_df[
    countries_df["country"] != "Unknown"
]

countries_df.head()

,show_id,country
0,s1,United States
1,s2,South Africa
4,s5,India
7,s8,United States
7,s8,Ghana


In [15]:
countries_df.to_sql(
    "netflix_countries",
    conn,
    if_exists="replace",
    index=False
)

print("Country table created successfully!")

Country table created successfully!


In [16]:
query = """
SELECT
    country,
    COUNT(*) AS total_titles
FROM netflix_countries
GROUP BY country
ORDER BY total_titles DESC
LIMIT 10;
"""

pd.read_sql_query(query, conn)

,country,total_titles
0,United States,3689
1,India,1046
2,United Kingdom,804
3,Canada,445
4,France,393
5,Japan,318
6,Spain,232
7,South Korea,231
8,Germany,226
9,Mexico,169


## 8. Content Distribution by Genre

In [17]:
genres_df = df[["show_id", "listed_in"]].copy()

genres_df["listed_in"] = genres_df["listed_in"].str.split(", ")

genres_df = genres_df.explode("listed_in")

genres_df.rename(
    columns={"listed_in": "genre"},
    inplace=True
)

genres_df.head()

,show_id,genre
0,s1,Documentaries
1,s2,International TV Shows
1,s2,TV Dramas
1,s2,TV Mysteries
2,s3,Crime TV Shows


In [18]:
genres_df.to_sql(
    "netflix_genres",
    conn,
    if_exists="replace",
    index=False
)

print("Genre table created successfully!")

Genre table created successfully!


In [19]:
query = """
SELECT
    genre,
    COUNT(*) AS total_titles
FROM netflix_genres
GROUP BY genre
ORDER BY total_titles DESC
LIMIT 10;
"""

pd.read_sql_query(query, conn)

,genre,total_titles
0,International Movies,2752
1,Dramas,2427
2,Comedies,1674
3,International TV Shows,1351
4,Documentaries,869
5,Action & Adventure,859
6,TV Dramas,763
7,Independent Movies,756
8,Children & Family Movies,641
9,Romantic Movies,616


## 9. JOIN Analysis — Content by Country and Type

In [20]:
query = """
SELECT
    c.country,
    n.type,
    COUNT(*) AS total_titles
FROM netflix_titles n
JOIN netflix_countries c
    ON n.show_id = c.show_id
GROUP BY c.country, n.type
ORDER BY total_titles DESC
LIMIT 20;
"""

pd.read_sql_query(query, conn)

,country,type,total_titles
0,United States,Movie,2751
1,India,Movie,962
2,United States,TV Show,938
3,United Kingdom,Movie,532
4,Canada,Movie,319
5,France,Movie,303
6,United Kingdom,TV Show,272
7,Japan,TV Show,199
8,Germany,Movie,182
9,Spain,Movie,171


In [21]:
query = """
SELECT
    rating,
    COUNT(*) AS total_titles
FROM netflix_titles
GROUP BY rating
HAVING COUNT(*) > 500
ORDER BY total_titles DESC;
"""

pd.read_sql_query(query, conn)

,rating,total_titles
0,TV-MA,3207
1,TV-14,2160
2,TV-PG,863
3,R,799


In [22]:
query = """
SELECT
    title,
    release_year,
    CASE
        WHEN release_year >= 2020 THEN 'Recent'
        WHEN release_year >= 2010 THEN 'Modern'
        ELSE 'Older'
    END AS content_category
FROM netflix_titles
LIMIT 20;
"""

pd.read_sql_query(query, conn)

,title,release_year,content_category
0,Dick Johnson Is Dead,2020,Recent
1,Blood & Water,2021,Recent
2,Ganglands,2021,Recent
3,Jailbirds New Orleans,2021,Recent
4,Kota Factory,2021,Recent
5,Midnight Mass,2021,Recent
6,My Little Pony: A New Generation,2021,Recent
7,Sankofa,1993,Older
8,The Great British Baking Show,2021,Recent
9,The Starling,2021,Recent


In [23]:
query = """
WITH country_counts AS (
    SELECT
        country,
        COUNT(*) AS total_titles
    FROM netflix_countries
    GROUP BY country
)

SELECT
    country,
    total_titles
FROM country_counts
ORDER BY total_titles DESC
LIMIT 10;
"""

pd.read_sql_query(query, conn)

,country,total_titles
0,United States,3689
1,India,1046
2,United Kingdom,804
3,Canada,445
4,France,393
5,Japan,318
6,Spain,232
7,South Korea,231
8,Germany,226
9,Mexico,169


In [24]:
query = """
WITH country_counts AS (
    SELECT
        country,
        COUNT(*) AS total_titles
    FROM netflix_countries
    GROUP BY country
)

SELECT
    country,
    total_titles,
    RANK() OVER (
        ORDER BY total_titles DESC
    ) AS country_rank
FROM country_counts
ORDER BY country_rank
LIMIT 10;
"""

pd.read_sql_query(query, conn)

,country,total_titles,country_rank
0,United States,3689,1
1,India,1046,2
2,United Kingdom,804,3
3,Canada,445,4
4,France,393,5
5,Japan,318,6
6,Spain,232,7
7,South Korea,231,8
8,Germany,226,9
9,Mexico,169,10


## 14. Subquery — Titles Released After Average Release Year

In [25]:
query = """
SELECT
    title,
    type,
    release_year
FROM netflix_titles
WHERE release_year > (
    SELECT AVG(release_year)
    FROM netflix_titles
)
ORDER BY release_year DESC
LIMIT 20;
"""

pd.read_sql_query(query, conn)

,title,type,release_year
0,Blood & Water,TV Show,2021
1,Ganglands,TV Show,2021
2,Jailbirds New Orleans,TV Show,2021
3,Kota Factory,TV Show,2021
4,Midnight Mass,TV Show,2021
5,My Little Pony: A New Generation,Movie,2021
6,The Great British Baking Show,TV Show,2021
7,The Starling,Movie,2021
8,"Vendetta: Truth, Lies and The Mafia",TV Show,2021
9,Bangkok Breaking,TV Show,2021


In [27]:
query = """
SELECT AVG(release_year)
FROM netflix_titles;
"""

pd.read_sql_query(query, conn)

,AVG(release_year)
0,2014.180198


In [29]:
query = """
SELECT
    title,
    type,
    release_year
FROM netflix_titles
WHERE release_year > (
    SELECT AVG(release_year)
    FROM netflix_titles
)
ORDER BY release_year DESC
LIMIT 20;
"""

pd.read_sql_query(query, conn)

,title,type,release_year
0,Blood & Water,TV Show,2021
1,Ganglands,TV Show,2021
2,Jailbirds New Orleans,TV Show,2021
3,Kota Factory,TV Show,2021
4,Midnight Mass,TV Show,2021
5,My Little Pony: A New Generation,Movie,2021
6,The Great British Baking Show,TV Show,2021
7,The Starling,Movie,2021
8,"Vendetta: Truth, Lies and The Mafia",TV Show,2021
9,Bangkok Breaking,TV Show,2021


In [30]:
query = """
WITH country_counts AS (
    SELECT
        country,
        COUNT(*) AS total_titles
    FROM netflix_countries
    GROUP BY country
)

SELECT
    country,
    total_titles,
    DENSE_RANK() OVER (
        ORDER BY total_titles DESC
    ) AS country_rank
FROM country_counts
ORDER BY country_rank
LIMIT 15;
"""

pd.read_sql_query(query, conn)

,country,total_titles,country_rank
0,United States,3689,1
1,India,1046,2
2,United Kingdom,804,3
3,Canada,445,4
4,France,393,5
5,Japan,318,6
6,Spain,232,7
7,South Korea,231,8
8,Germany,226,9
9,Mexico,169,10


## 16. Year-wise Movies vs TV Shows

In [31]:
query = """
SELECT
    release_year,
    SUM(CASE WHEN type = 'Movie' THEN 1 ELSE 0 END) AS movies,
    SUM(CASE WHEN type = 'TV Show' THEN 1 ELSE 0 END) AS tv_shows
FROM netflix_titles
GROUP BY release_year
ORDER BY release_year DESC
LIMIT 20;
"""

pd.read_sql_query(query, conn)

,release_year,movies,tv_shows
0,2021,277,315
1,2020,517,436
2,2019,633,397
3,2018,767,380
4,2017,767,265
5,2016,658,244
6,2015,398,162
7,2014,264,88
8,2013,225,63
9,2012,173,64


## 17. Rating Distribution by Content Type

In [ ]:
query = """
SELECT
    rating,
    SUM(CASE WHEN type = 'Movie' THEN 1 ELSE 0 END) AS movies,
    SUM(CASE WHEN type = 'TV Show' THEN 1 ELSE 0 END) AS tv_shows
FROM netflix_titles
GROUP BY rating
ORDER BY movies DESC;
"""

pd.read_sql_query(query, conn)